In [6]:
pip install gensim scikit-learn tabulate numpy


Defaulting to user installation because normal site-packages is not writeable
Looking in links: /usr/share/pip-wheels
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 27.9/27.9 MB 11.0 MB/s  0:00:02eta 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2/2 [gensim]━━━━━ 1/2 [gensim]
Note: you may need to restart the kernel to use updated packages.


In [7]:
# Step 1: Import libraries
import numpy as np
from sklearn.cluster import KMeans
from gensim.models import Word2Vec
from tabulate import tabulate
from collections import Counter

# Step 2: Create the documents
dataset = [
    "I love playing football on the weekends",
    "I enjoy hiking and camping in the mountains",
    "I like to read books and watch movies",
    "I prefer playing video games over sports",
    "I love listening to music and going to concerts"
]

# Step 3: Train Word2Vec model
tokenized_dataset = [doc.lower().split() for doc in dataset]

word2vec_model = Word2Vec(
    sentences=tokenized_dataset,
    vector_size=100,
    window=5,
    min_count=1,
    workers=4
)

# Step 4: Create document embeddings (average word vectors)
X = np.array([
    np.mean(
        [word2vec_model.wv[word] for word in doc if word in word2vec_model.wv],
        axis=0
    )
    for doc in tokenized_dataset
])

# Step 5: Perform clustering
k = 2
km = KMeans(n_clusters=k, random_state=42, n_init=10)
km.fit(X)

# Predict clusters
y_pred = km.labels_

# Display results
table_data = [["Document", "Predicted Cluster"]]
for doc, cluster in zip(dataset, y_pred):
    table_data.append([doc, cluster])

print(tabulate(table_data, headers="firstrow", tablefmt="grid"))

# Step 6: Evaluate results (Purity)
total_samples = len(y_pred)
purity_sum = 0

for cluster_id in range(k):
    cluster_indices = [i for i in range(len(y_pred)) if y_pred[i] == cluster_id]
    cluster_labels = [y_pred[i] for i in cluster_indices]
    counts = Counter(cluster_labels)
    purity_sum += max(counts.values())

purity = purity_sum / total_samples
print("Purity:", purity)

+-------------------------------------------------+---------------------+
| Document                                        |   Predicted Cluster |
+=================================================+=====================+
| I love playing football on the weekends         |                   1 |
+-------------------------------------------------+---------------------+
| I enjoy hiking and camping in the mountains     |                   1 |
+-------------------------------------------------+---------------------+
| I like to read books and watch movies           |                   0 |
+-------------------------------------------------+---------------------+
| I prefer playing video games over sports        |                   1 |
+-------------------------------------------------+---------------------+
| I love listening to music and going to concerts |                   0 |
+-------------------------------------------------+---------------------+
Purity: 1.0


/opt/conda/envs/anaconda-2025.12-py312/lib/python3.12/site-packages/joblib/externals/loky/backend/context.py:131: UserWarning: Could not find the number of physical cores for the following reason:
found 0 physical cores < 1
Returning the number of logical cores instead. You can silence this warning by setting LOKY_MAX_CPU_COUNT to the number of cores you want to use.
  warnings.warn(
  File "/opt/conda/envs/anaconda-2025.12-py312/lib/python3.12/site-packages/joblib/externals/loky/backend/context.py", line 255, in _count_physical_cores
    raise ValueError(f"found {cpu_count_physical} physical cores < 1")
